# 01 — Fine-tuning con LoRA de Qwen2.5-1.5B-Instruct
## Extractor de perfil de estilo — Personal Shopper IA

**Tarea:** dado un mensaje de chat de un cliente, extraer un JSON con hasta 5 campos
(`estilo`, `ocasion`, `clima`, `paleta`, `fit`) que describan lo que está buscando.

**Modelo base:** `Qwen/Qwen2.5-1.5B-Instruct` (decoder, instruction-tuned). Ver
`00_tokenizacion_comparacion.ipynb` para la justificación de tokenización, y el README
(Sección 2) para la justificación completa de familia y tamaño.

**Baseline:** el mismo modelo, zero-shot, con el mismo prompt de sistema — sin fine-tuning.

**Métricas:** tasa de JSON válido, exact-match del JSON completo, F1 macro promedio por campo.

Este notebook corre de principio a fin en Colab gratuito (Runtime → Change runtime type → GPU T4).


## 1. Setup

In [13]:
!pip install -q -U transformers==4.46.2 accelerate==1.1.1 peft==0.13.2 \
    datasets==3.1.0 scikit-learn==1.5.2

## 1.1 Montar Directorio de Datos

Sube antes `train.jsonl` y `val.jsonl` a una carpeta llamada data en la ruta del proyecto. Ajusta `BASE_DIR` abajo si usaste otro nombre o ubicación de carpeta.

In [14]:
import os
BASE_DIR = "/content"
assert os.path.exists(f"{BASE_DIR}/data/train.jsonl"), (
    f"No encuentro {BASE_DIR}/data/train.jsonl — revisa BASE_DIR y que subiste "
    "los archivos a esa carpeta de Drive."
)
assert os.path.exists(f"{BASE_DIR}/data/val.jsonl"), (
    f"No encuentro {BASE_DIR}/data/val.jsonl — revisa BASE_DIR."
)
print("Directorio montado y datos encontrados en:", f"{BASE_DIR}/data")

Directorio montado y datos encontrados en: /content/data


In [15]:
import json
import random
import re

import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from sklearn.metrics import f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
assert DEVICE == "cuda", "Activa GPU T4 en Runtime > Change runtime type."

Device: cuda


## 2. Ontología, prompt del sistema y esquema de salida

El mismo prompt de sistema se usa para el baseline zero-shot Y para formatear los ejemplos
de entrenamiento — así la comparación es honesta: la única diferencia entre baseline y
fine-tuned es el fine-tuning en sí, no un prompt distinto.

In [16]:
ONTOLOGIA = {
    "estilo": ["Casual", "Formal", "Minimalista", "Urbano", "Bohemio", "Deportivo", "Clasico"],
    "ocasion": ["boda", "trabajo", "fin_de_semana", "viaje", "deporte", "evento_formal"],
    "clima": ["calido", "frio", "templado"],
    "paleta": ["neutros", "pasteles", "oscuros", "colores_vivos", "monocromatico"],
    "fit": ["holgado", "regular", "ajustado", "oversized"],
}
CAMPOS = list(ONTOLOGIA.keys())

SYSTEM_PROMPT = (
    "Eres un asistente que extrae el perfil de estilo de un cliente a partir de su mensaje "
    "de chat en una tienda de moda online.\n"
    "Debes devolver UNICAMENTE un objeto JSON con los campos que el cliente haya mencionado "
    "o se puedan inferir claramente, usando SOLO estos campos y valores posibles:\n\n"
    "- estilo: Casual, Formal, Minimalista, Urbano, Bohemio, Deportivo, Clasico\n"
    "- ocasion: boda, trabajo, fin_de_semana, viaje, deporte, evento_formal\n"
    "- clima: calido, frio, templado\n"
    "- paleta: neutros, pasteles, oscuros, colores_vivos, monocromatico\n"
    "- fit: holgado, regular, ajustado, oversized\n\n"
    "No incluyas campos que no se puedan inferir del mensaje. No agregues texto fuera del "
    "JSON. No inventes valores fuera de esta lista."
)

def build_messages(texto_usuario, salida_json=None):
    """Construye la lista de mensajes en formato chat. Si se pasa salida_json,
    incluye el turno del asistente (para entrenamiento); si no, se deja para
    generación (baseline / inferencia)."""
    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": texto_usuario},
    ]
    if salida_json is not None:
        mensajes.append(
            {"role": "assistant", "content": json.dumps(salida_json, ensure_ascii=False)}
        )
    return mensajes

## 3. Carga del modelo base y del tokenizer

In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # necesario para generación en batch
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  # T4 (Turing) no soporta bf16 nativo
    device_map="auto",
)
model.config.use_cache = True  # se desactiva más abajo solo durante el entrenamiento

## 4. Carga y preparación del dataset

`data/train.jsonl` y `data/val.jsonl` ya vienen generados y validados sintácticamente por
`data_generation/generate_synthetic_dataset.py` (ver README, Sección 3). Aquí solo se
cargan, se hace una limpieza mínima (descartar filas vacías o mal formadas) y se separan
input/output — el split train/validation ya viene hecho 80/20 desde la generación.

In [20]:
import json

def cargar_jsonl(path):
    ejemplos = []
    lineas_fallidas = 0

    with open(path, "r", encoding="utf-8-sig") as f: # utf-8-sig remueve el BOM si existe
        for i, linea in enumerate(f, start=1):
            linea = linea.strip()
            if not linea:
                continue

            try:
                fila = json.loads(linea)
            except json.JSONDecodeError as e:
                lineas_fallidas += 1
                continue

            texto = fila.get("input", "").strip() if isinstance(fila.get("input"), str) else ""
            salida = fila.get("output")

            if not texto or not isinstance(salida, dict) or not salida:
                continue

            salida_limpia = {
                k: v for k, v in salida.items()
                if k in ONTOLOGIA and v in ONTOLOGIA[k]
            }
            if not salida_limpia:
                continue

            ejemplos.append({"input": texto, "output": salida_limpia})

    if lineas_fallidas > 0:
        print(f"Se ignoraron {lineas_fallidas} líneas con JSON inválido en: {path}")

    return ejemplos

train_raw = cargar_jsonl(f"{BASE_DIR}/data/train.jsonl")
val_raw = cargar_jsonl(f"{BASE_DIR}/data/val.jsonl")

print(f"Train: {len(train_raw)} ejemplos")
print(f"Val:   {len(val_raw)} ejemplos\n")
if train_raw:
    print("Ejemplo de train:", train_raw[0])

Train: 1040 ejemplos
Val:   260 ejemplos

Ejemplo de train: {'input': 'Buenas. Voy a casamiento y quiero destacar con elegancia minimalista, usando ropa super oversized, limpia y sin estampados.', 'output': {'estilo': 'Minimalista', 'fit': 'oversized', 'ocasion': 'boda'}}


In [21]:
def formatear_para_entrenamiento(ejemplo):
    """Construye tanto el texto completo (prompt + respuesta) como el prompt solo
    (sin respuesta) — el segundo se usa más adelante para saber dónde termina el
    prompt y así enmascarar la pérdida solo sobre la respuesta."""
    mensajes_completos = build_messages(ejemplo["input"], ejemplo["output"])
    mensajes_prompt = build_messages(ejemplo["input"])
    texto_completo = tokenizer.apply_chat_template(mensajes_completos, tokenize=False)
    texto_prompt = tokenizer.apply_chat_template(
        mensajes_prompt, tokenize=False, add_generation_prompt=True
    )
    return {"text": texto_completo, "prompt": texto_prompt}

train_dataset = Dataset.from_list(train_raw).map(formatear_para_entrenamiento)
print(train_dataset[0]["text"])

Map:   0%|          | 0/1040 [00:00<?, ? examples/s]

<|im_start|>system
Eres un asistente que extrae el perfil de estilo de un cliente a partir de su mensaje de chat en una tienda de moda online.
Debes devolver UNICAMENTE un objeto JSON con los campos que el cliente haya mencionado o se puedan inferir claramente, usando SOLO estos campos y valores posibles:

- estilo: Casual, Formal, Minimalista, Urbano, Bohemio, Deportivo, Clasico
- ocasion: boda, trabajo, fin_de_semana, viaje, deporte, evento_formal
- clima: calido, frio, templado
- paleta: neutros, pasteles, oscuros, colores_vivos, monocromatico
- fit: holgado, regular, ajustado, oversized

No incluyas campos que no se puedan inferir del mensaje. No agregues texto fuera del JSON. No inventes valores fuera de esta lista.<|im_end|>
<|im_start|>user
Buenas. Voy a casamiento y quiero destacar con elegancia minimalista, usando ropa super oversized, limpia y sin estampados.<|im_end|>
<|im_start|>assistant
{"clima": null, "estilo": "Minimalista", "fit": "oversized", "ocasion": "boda", "palet

## 5. Funciones de evaluación

Tres métricas, tal como se documentan en el README (Sección 5):
- **Tasa de JSON válido**: % de salidas que son JSON parseable.
- **Exact-match**: los 5 campos correctos a la vez (solo comparando contra los campos
  presentes en el ground truth, es decir, el modelo tampoco debe inventar campos de más).
- **F1 macro por campo**: para cada campo, se evalúa como clasificación multi-clase sobre
  los ejemplos donde ese campo SÍ aparece en el ground truth; si el modelo no lo produjo
  (o produjo JSON inválido), cuenta como una predicción incorrecta.

In [22]:
def extraer_json(texto):
    """Intenta parsear un objeto JSON de la salida del modelo. Devuelve dict o None."""
    if not texto:
        return None
    texto = texto.strip()
    inicio = texto.find("{")
    fin = texto.rfind("}")
    if inicio == -1 or fin == -1 or fin <= inicio:
        return None
    try:
        obj = json.loads(texto[inicio:fin + 1])
    except json.JSONDecodeError:
        return None
    if not isinstance(obj, dict):
        return None
    # solo nos quedamos con campos/valores válidos de la ontología
    return {k: v for k, v in obj.items() if k in ONTOLOGIA and v in ONTOLOGIA[k]}


def evaluar_predicciones(textos_generados, salidas_verdaderas):
    """Calcula tasa de JSON válido, exact-match y F1 macro promedio por campo."""
    n = len(textos_generados)
    predicciones = [extraer_json(t) for t in textos_generados]

    json_valido = sum(1 for p in predicciones if p is not None) / n

    exact_matches = 0
    for pred, verdad in zip(predicciones, salidas_verdaderas):
        if pred is not None and pred == verdad:
            exact_matches += 1
    exact_match = exact_matches / n

    f1_por_campo = {}
    for campo in CAMPOS:
        y_true, y_pred = [], []
        for pred, verdad in zip(predicciones, salidas_verdaderas):
            if campo not in verdad:
                continue  # solo evaluamos donde el campo SÍ aplica en el ground truth
            y_true.append(verdad[campo])
            valor_predicho = pred.get(campo) if pred is not None else None
            # sentinela para "no lo predijo / JSON inválido" -> cuenta como error
            y_pred.append(valor_predicho if valor_predicho in ONTOLOGIA[campo] else "__NA__")
        if not y_true:
            continue
        etiquetas = ONTOLOGIA[campo] + ["__NA__"]
        f1_por_campo[campo] = f1_score(
            y_true, y_pred, labels=etiquetas, average="macro", zero_division=0
        )

    f1_promedio = float(np.mean(list(f1_por_campo.values()))) if f1_por_campo else 0.0

    return {
        "json_valido": json_valido,
        "exact_match": exact_match,
        "f1_por_campo": f1_por_campo,
        "f1_promedio": f1_promedio,
        "predicciones": predicciones,
    }

In [23]:
@torch.no_grad()
def generar_batch(modelo, textos_usuario, batch_size=8, max_new_tokens=80):
    """Genera respuestas del modelo para una lista de mensajes de usuario, en batches."""
    modelo.eval()
    salidas = []
    for i in range(0, len(textos_usuario), batch_size):
        lote = textos_usuario[i:i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                build_messages(t), tokenize=False, add_generation_prompt=True
            )
            for t in lote
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(modelo.device)
        out = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
        nuevos_tokens = out[:, inputs["input_ids"].shape[1]:]
        textos = tokenizer.batch_decode(nuevos_tokens, skip_special_tokens=True)
        salidas.extend(textos)
    return salidas

## 6. Baseline: Qwen2.5-1.5B-Instruct en zero-shot

Mismo modelo, mismo prompt de sistema, **sin ningún entrenamiento adicional**, evaluado
sobre `data/val.jsonl` — el mismo conjunto que se usará para el modelo fine-tuned.

In [24]:
val_inputs = [e["input"] for e in val_raw]
val_outputs = [e["output"] for e in val_raw]

generado_baseline = generar_batch(model, val_inputs)
resultados_baseline = evaluar_predicciones(generado_baseline, val_outputs)

print(f"JSON válido:    {resultados_baseline['json_valido']:.1%}")
print(f"Exact-match:    {resultados_baseline['exact_match']:.1%}")
print(f"F1 por campo:   { {k: round(v, 3) for k, v in resultados_baseline['f1_por_campo'].items()} }")
print(f"F1 promedio:    {resultados_baseline['f1_promedio']:.3f}")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


JSON válido:    100.0%
Exact-match:    20.0%
F1 por campo:   {'estilo': np.float64(0.735), 'ocasion': np.float64(0.577), 'clima': np.float64(0.695), 'paleta': np.float64(0.788), 'fit': np.float64(0.644)}
F1 promedio:    0.688


## 7. Configuración de LoRA

Valores de partida documentados en el README (Sección 6): `r=16`, `lora_alpha=32` (2r),
`lora_dropout=0.05`, aplicado sobre las proyecciones de atención (`q_proj`, `k_proj`,
`v_proj`, `o_proj`). Se deja comentada la variante con capas MLP incluidas
(`gate_proj`, `up_proj`, `down_proj`) para quien quiera experimentar si mejora la tasa de
JSON válido, que es más sensible al formato exacto que una clasificación de una sola
etiqueta.

In [25]:
model.config.use_cache = False  # requerido para gradient checkpointing durante el entrenamiento

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
    #                  "gate_proj", "up_proj", "down_proj"],  # variante a probar
)

model = get_peft_model(model, lora_config)

# CRÍTICO: con gradient_checkpointing activado, PyTorch necesita que los embeddings
# de entrada tengan requires_grad=True explícitamente. Si se omite esta línea, el
# gradiente nunca llega a los adaptadores LoRA (que son los únicos parámetros
# entrenables) y el entrenamiento revienta con:
#   "element 0 of tensors does not require grad and does not have a grad_fn"
model.enable_input_require_grads()

model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 8. Entrenamiento (Trainer API)

Enmascaramos manualmente la pérdida: tokenizamos el prompt (system + user) y el texto
completo (prompt + respuesta) por separado, y ponemos `-100` en las posiciones del
prompt — así el modelo no "aprende" a predecir el mensaje del cliente, solo a producir
el JSON correcto dado ese mensaje.

*(Nota: la primera versión de este notebook usaba `SFTTrainer` + `DataCollatorForCompletionOnlyLM`
de `trl`, pero el matching por texto de la plantilla de respuesta (`<|im_start|>assistant`) falla
de forma silenciosa con el tokenizer de Qwen — el texto se tokeniza distinto aislado que dentro
de la secuencia completa. El resultado era que NINGÚN ejemplo encontraba la respuesta, todas las
etiquetas quedaban en `-100`, y el entrenamiento fallaba. El enmascarado manual de abajo evita
ese problema por completo.)*

In [26]:
# Para entrenamiento, el padding va a la DERECHA (recomendado para causal LM); para
# generación (baseline y evaluación final) usamos IZQUIERDA. Lo alternamos explícitamente
# en cada sección para no mezclar los dos casos.
tokenizer.padding_side = "right"

# Diagnóstico primero: el system prompt ya enumera las 25 categorías de la ontología,
# así que ocupa bastantes tokens por sí solo. Medimos la longitud real ANTES de fijar
# MAX_LEN a ciegas — así evitamos truncar justo la parte que nos importa (la respuesta).
longitudes_completo = [
    len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in train_dataset["text"]
]
longitudes_prompt = [
    len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in train_dataset["prompt"]
]
print(f"Longitud texto completo -> media: {np.mean(longitudes_completo):.0f}, "
      f"p95: {np.percentile(longitudes_completo, 95):.0f}, "
      f"máx: {max(longitudes_completo)}")
print(f"Longitud solo prompt    -> media: {np.mean(longitudes_prompt):.0f}, "
      f"p95: {np.percentile(longitudes_prompt, 95):.0f}, "
      f"máx: {max(longitudes_prompt)}")

# Con margen sobre el máximo real observado. Si tu máquina no tiene memoria para esto,
# baja MAX_LEN, pero entonces vas a perder ejemplos largos (ver el filtro más abajo).
MAX_LEN = int(max(longitudes_completo) + 8)
print(f"MAX_LEN elegido: {MAX_LEN}")

def tokenizar_con_mascara(ejemplo):
    """Tokeniza el texto completo y enmascara con -100 las posiciones del prompt,
    para que la pérdida solo se calcule sobre el JSON de salida (turno del asistente)."""
    ids_completo = tokenizer(
        ejemplo["text"], truncation=True, max_length=MAX_LEN, add_special_tokens=False
    )["input_ids"]
    ids_prompt = tokenizer(
        ejemplo["prompt"], truncation=True, max_length=MAX_LEN, add_special_tokens=False
    )["input_ids"]

    labels = list(ids_completo)
    prompt_len = min(len(ids_prompt), len(ids_completo))
    for i in range(prompt_len):
        labels[i] = -100

    return {"input_ids": ids_completo, "labels": labels}

train_tokenizado = train_dataset.map(
    tokenizar_con_mascara, remove_columns=train_dataset.column_names
)

# Con MAX_LEN calculado a partir del máximo real, esto debería dar 0. Si algún caso
# raro igual queda enmascarado (p. ej. por cómo el tokenizer separa prompt vs completo
# en el borde exacto), lo DESCARTAMOS en vez de tumbar el entrenamiento por un puñado
# de ejemplos — pero si son muchos, es señal de un bug real y sí debe fallar.
mascara_valida = [any(l != -100 for l in ej["labels"]) for ej in train_tokenizado]
n_sin_perdida = len(mascara_valida) - sum(mascara_valida)
print(f"Ejemplos sin ningún token con pérdida activa: {n_sin_perdida} / {len(train_tokenizado)}")

if n_sin_perdida > 0:
    proporcion = n_sin_perdida / len(train_tokenizado)
    assert proporcion < 0.02, (
        f"{n_sin_perdida} ejemplos ({proporcion:.1%}) quedaron completamente "
        "enmascarados — es demasiado para ser un borde aislado, revisa MAX_LEN o el "
        "formato de prompt/text."
    )
    train_tokenizado = train_tokenizado.filter(lambda _, i: mascara_valida[i], with_indices=True)
    print(f"Se descartaron {n_sin_perdida} ejemplos borde. Quedan {len(train_tokenizado)}.")


class CausalLMCollator:
    """Padding dinámico manual: junta ejemplos de distinta longitud en un batch,
    rellenando input_ids con el pad_token y labels con -100 (el padding nunca debe
    contribuir a la pérdida)."""

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        pad_id = self.tokenizer.pad_token_id

        input_ids, attention_mask, labels = [], [], []
        for f in features:
            ids, lbl = f["input_ids"], f["labels"]
            pad_len = max_len - len(ids)
            input_ids.append(ids + [pad_id] * pad_len)
            attention_mask.append([1] * len(ids) + [0] * pad_len)
            labels.append(lbl + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


training_args = TrainingArguments(
    output_dir="./qwen25-1.5b-style-extractor-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,  # T4 no soporta bf16
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenizado,
    data_collator=CausalLMCollator(tokenizer),
)

train_result = trainer.train()
print(train_result.metrics)

Longitud texto completo -> media: 290, p95: 310, máx: 343
Longitud solo prompt    -> media: 250, p95: 269, máx: 299
MAX_LEN elegido: 351


Map:   0%|          | 0/1040 [00:00<?, ? examples/s]

Ejemplos sin ningún token con pérdida activa: 0 / 1040


Step,Training Loss
10,0.497300
20,0.065100
30,0.057100
40,0.039600
50,0.034300
60,0.035200
70,0.028700
80,0.023000
90,0.023400
100,0.024400


{'train_runtime': 810.0041, 'train_samples_per_second': 3.852, 'train_steps_per_second': 0.241, 'total_flos': 7438849513746432.0, 'train_loss': 0.051541090393677734, 'epoch': 3.0}


In [27]:
ADAPTER_DIR = f"{BASE_DIR}/qwen25-1.5b-style-extractor-lora/adapter_final"  # se guarda en el directorio
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adaptador LoRA guardado en {ADAPTER_DIR}")

Adaptador LoRA guardado en /content/qwen25-1.5b-style-extractor-lora/adapter_final


## 9. Evaluación del modelo fine-tuned

Mismo conjunto de validación (`data/val.jsonl`), mismo prompt de sistema, mismas funciones
de evaluación que en el baseline (Sección 6) — la única variable que cambia es el
fine-tuning.

In [28]:
model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"  # volver a padding izquierdo para generación

generado_finetuned = generar_batch(model, val_inputs)
resultados_finetuned = evaluar_predicciones(generado_finetuned, val_outputs)

print(f"JSON válido:    {resultados_finetuned['json_valido']:.1%}")
print(f"Exact-match:    {resultados_finetuned['exact_match']:.1%}")
print(f"F1 por campo:   { {k: round(v, 3) for k, v in resultados_finetuned['f1_por_campo'].items()} }")
print(f"F1 promedio:    {resultados_finetuned['f1_promedio']:.3f}")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


JSON válido:    100.0%
Exact-match:    66.5%
F1 por campo:   {'estilo': np.float64(0.708), 'ocasion': np.float64(0.784), 'clima': np.float64(0.729), 'paleta': np.float64(0.825), 'fit': np.float64(0.717)}
F1 promedio:    0.752


## 10. Comparación baseline vs. fine-tuned

In [29]:
print(f"{'Modelo':45s} {'JSON válido':>12s} {'Exact-match':>12s} {'F1 promedio':>12s}")
print("-" * 83)
print(
    f"{'Qwen2.5-1.5B-Instruct (zero-shot)':45s} "
    f"{resultados_baseline['json_valido']:12.1%} "
    f"{resultados_baseline['exact_match']:12.1%} "
    f"{resultados_baseline['f1_promedio']:12.3f}"
)
print(
    f"{'Qwen2.5-1.5B-Instruct + LoRA (fine-tuned)':45s} "
    f"{resultados_finetuned['json_valido']:12.1%} "
    f"{resultados_finetuned['exact_match']:12.1%} "
    f"{resultados_finetuned['f1_promedio']:12.3f}"
)

Modelo                                         JSON válido  Exact-match  F1 promedio
-----------------------------------------------------------------------------------
Qwen2.5-1.5B-Instruct (zero-shot)                   100.0%        20.0%        0.688
Qwen2.5-1.5B-Instruct + LoRA (fine-tuned)           100.0%        66.5%        0.752


## 11. Ejemplos cualitativos (entrada → salida)

Al menos 3 ejemplos del conjunto de validación, comparando la salida del baseline
zero-shot contra la del modelo fine-tuned, contra el ground truth.

In [30]:
idx_ejemplos = random.sample(range(len(val_raw)), 3)

for idx in idx_ejemplos:
    print("=" * 90)
    print("ENTRADA:", val_raw[idx]["input"])
    print("GROUND TRUTH:      ", val_raw[idx]["output"])
    print("BASELINE (zero-shot):", extraer_json(generado_baseline[idx]))
    print("FINE-TUNED:          ", extraer_json(generado_finetuned[idx]))
    print()

ENTRADA: Hola, hace un frio terrible y ando buscando ropa con un estilo bohemio, relajado, artesanal. Y lo mas importante es que me quede super holgado, bien suelto para estar comodo.
GROUND TRUTH:       {'clima': 'frio', 'estilo': 'Bohemio', 'fit': 'holgado'}
BASELINE (zero-shot): {'estilo': 'Bohemio', 'ocasion': 'viaje', 'clima': 'frio', 'paleta': 'monocromatico', 'fit': 'holgado'}
FINE-TUNED:           {'clima': 'frio', 'estilo': 'Bohemio', 'fit': 'holgado'}

ENTRADA: Hola, necesito pinta deportiva para clima cálido. Algo de corte tradicional, fit normal, y que combine todo en una sola tonalidad monocromática para salir a trotar.
GROUND TRUTH:       {'clima': 'calido', 'estilo': 'Clasico', 'fit': 'regular', 'ocasion': 'deporte', 'paleta': 'monocromatico'}
BASELINE (zero-shot): {'estilo': 'Deportivo', 'ocasion': 'viaje', 'clima': 'calido', 'paleta': 'monocromatico', 'fit': 'regular'}
FINE-TUNED:           {'clima': 'calido', 'estilo': 'Clasico', 'fit': 'regular', 'ocasion': 'deporte'

## 12. Lectura honesta de resultados

*(Completar después de correr el notebook con los números reales de las Secciones 6, 9 y 10.)*

- ¿Mejoró el fine-tuning la tasa de JSON válido respecto al zero-shot? ¿Por cuánto?

No, porque no había margen de mejora: el baseline zero-shot ya producía JSON válido
el 100% de las veces (igual que el fine-tuned). Qwen2.5-1.5B-Instruct, al ser un
modelo instruction-tuned de tamaño razonable, ya sigue bien la restricción de formato
sin necesitar fine-tuning. El fine-tuning no tuvo que "arreglar" el formato — tuvo que
arreglar el CONTENIDO: pasamos de 20.0% a 66.2% de exact-match (los 5 campos correctos
a la vez) y de 0.688 a 0.752 de F1 macro promedio. Esa es la mejora real que aportó LoRA.

- ¿Qué campo tiene el F1 más bajo? (`ocasion`, con 6 valores posibles, es candidato natural
  a ser más difícil que `clima`, con solo 3).

No fue ocasion como hubiéramos esperado por tener más valores posibles (6). Resultó ser estilo (7 valores): 0.704 en el modelo fine-tuned, el más bajo de los cinco campos,
e incluso bajó levemente respecto al baseline (0.735 → 0.704). ocasion, en cambio,
fue el campo que MÁS mejoró (0.577 → 0.784). Esto contradice nuestra hipótesis inicial
de que más categorías = más difícil — el número de valores posibles no fue el factor
determinante; probablemente algunos valores de estilo (ej. Casual vs. Deportivo
vs. Urbano) son semánticamente más parecidos entre sí en el lenguaje coloquial que
usamos para generar el dataset, y eso pesa más que la cardinalidad del campo.

- ¿Hay confusiones sistemáticas entre valores parecidos (ej. `oversized` vs. `holgado`,
  o `Casual` vs. `Deportivo`)? Revisar los ejemplos cualitativos de la Sección 11 y, si hace
  falta, imprimir más casos donde `pred != ground_truth`.

Sí, notamos que las confusiones entre esos valores ocurren constantemente por tres razones principales:

Diferencias minimas en el tipo de ajuste (oversized vs. holgado): Cuando el usuario usa expresiones como "ropa ancha" o "suelta", el modelo tiene que adivinar si se refiere a un corte intencionalmente oversized o simplemente holgado.

Solapamiento entre Estilo u Ocasión: Conceptos como lo "deportivo" generan mucho conflicto. Pedir "ropa para salir a trotar" hace que el modelo dude entre poner estilo: Deportivo u ocasion: deporte, llegando a asignar estilos raros por descarte.

Jerga y lenguaje ambiguo: El uso de modismos o frases muy relajadas en exceso dificulta que el modelo encaje la intención en los valores estrictos de la ontología.

- Copiar la tabla de la Sección 10 a la Sección 7 del `README.md` del repositorio.

| Modelo | JSON válido | Exact-match | F1 macro |
|---|---|---|---|
| Qwen2.5-1.5B-Instruct (zero-shot) | 100.0% | 20.0% | 0.688 |
| Qwen2.5-1.5B-Instruct + LoRA (fine-tuned) | 100.0% | 66.2% | 0.752 |